In [ ]:
# ==============================================================================
# Purpose:
#   Compute ssGSEA/GSVA signature scores for the NHB-BC Up gene sets (from a GMT)
#   in external bulk expression datasets, then merge scores into metadata tables
#   for downstream plotting/association tests.
#
# Steps:
#   1) Load NHB signature gene sets from GMT (NHB-BC Up All + NHB-BC Up Protein).
#   2) For each external dataset folder:
#        - read cleaned metadata (samples × covariates)
#        - read cleaned expression matrix (genes × samples, already normalized)
#        - wrap expression + pheno into an ExpressionSet
#        - run GSVA in ssGSEA mode (via ssgseaParam + gsva())
#        - transpose scores to samples × gene-sets
#        - merge scores back into metadata by sample ID
#        - write merged table to CSV

# Header

In [1]:
library(GSVA)
library(GSEABase)
library(glue)

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, saveRDS, setdiff,
    table, tapply, union, unique, unsplit, which.max, which.min


Loading required package: Biobase

Welcome to Bioconductor

    Vignettes contain introductory material; view with
    'browseVignettes()'. To cite Bioconductor, see
    'citation("Biobase")', and for packages 'citation("pkgname")'.


Loading required package: annotate

Loading required package: AnnotationDbi

Loading required package: stats4

Loading required package: IRang

In [2]:
source('functions.R')

Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘sp’


The following object is masked from ‘package:IRanges’:

    %over%



Attaching package: ‘SeuratObject’


The following object is masked from ‘package:GSEABase’:

    intersect


The following object is masked from ‘package:IRanges’:

    intersect


The following object is masked from ‘package:S4Vectors’:

    intersect


The following object is masked from ‘package:BiocGenerics’:

    intersect


The following objects are masked from ‘package:base’:

    intersect, t



Attaching package: ‘patchwork’


The following object is masked from ‘package:XML’:

    free



Attaching package: ‘gridExtra’


The following object is masked from ‘package:Biobase’:

    combine


The following object is masked from ‘package:BiocGenerics’:

    combine



Attaching package: ‘dplyr’


The following object is masked from ‘package:gridExtra’:

    combine


The following objects are masked from ‘package:GS

In [4]:
setwd('../')

# Calc IMM-POP scores in IWSC dataset

In [20]:
pb = qs::qread(
    'data/06_pb/pb_donor_celltypelevel3.qs'
)
meta = pb@meta.data
dat = pb@assays[['RNA']]$data

In [15]:
gsets = parse_gmt('data/08_signature/IMMPOP.gmt')

In [16]:
pheno_data <- AnnotatedDataFrame(data = meta)
eset <- ExpressionSet(
  assayData = as.matrix(dat),  # Expression data matrix
  phenoData = pheno_data                   # Phenotype data
)

ssgsea = ssgseaParam(eset, gsets)
ssgsea_out = gsva(ssgsea)
ssgsea_out_t = t(exprs(ssgsea_out))

ERROR: Error in h(simpleError(msg, call)): error in evaluating the argument 'data' in selecting a method for function 'AnnotatedDataFrame': object 'meta' not found


In [17]:
head(ssgsea_out_t)

,IMM-POP Up,IMM-POP Dn
SRX9031276,0.8438967,1.1227429
SRX9031216,0.7892972,1.1984780
SRX9031279,0.9106433,1.0628143
SRX4780427,0.7189737,1.2707840
SRX8136180,0.7627995,1.2954935
SRX6419103,0.8125830,0.6341157


# Calc Sig Scores - IMM-POP

In [30]:
gmt_file <- "data/08_signature/IMMPOP.gmt" #"data/geneset/NHB_dn.gmt"
gsets <- getGmt(gmt_file)

## BC cohorts

In [31]:
meta3 = read.csv('data/09_externalds/23pt_clean/clean_meta.csv', row.names=1)
dat3 = read.csv('data/09_externalds/23pt_clean/clean_expr.csv', row.names=1)

In [32]:
pheno_data <- AnnotatedDataFrame(data = meta3)
eset <- ExpressionSet(
  assayData = as.matrix(dat3),  # Expression data matrix
  phenoData = pheno_data                   # Phenotype data
)

experimentData(eset) <- MIAME(
  name = "ex23",
)

In [33]:
ssgsea = ssgseaParam(eset, gsets)

ssgsea_out = gsva(ssgsea)
ssgsea_out_t = t(exprs(ssgsea_out))


ℹ GSVA version 2.0.7

! 9678 genes with constant values throughout the samples

! No annotation metadata available in the input expression data object

! Attempting to directly match identifiers in expression data to gene sets

ℹ Calculating  ssGSEA scores for 2 gene sets

ℹ Calculating ranks

ℹ Calculating rank weights

ℹ Normalizing ssGSEA scores

✔ Calculations finished



In [34]:
mg <- merge(meta3, ssgsea_out_t, by = "row.names")
rownames(mg) <- mg$Row.names
mg$Row.names <- NULL

write.csv(mg, 'data/09_externalds/23pt_clean/gsva_23pt_IMMPOP.csv') # 

## Pan-Ca Cohort

In [35]:
dpath = 'data/09_externalds/18k_clean'
meta3 = read.csv(glue('{dpath}/clean_meta.csv'), row.names=1)
dat3 = read.csv(glue('{dpath}/clean_expr.csv'), row.names=1)

In [36]:
pheno_data <- AnnotatedDataFrame(data = meta3)
eset <- ExpressionSet(
  assayData = as.matrix(dat3),  # Expression data matrix
  phenoData = pheno_data                   # Phenotype data
)

In [37]:
experimentData(eset) <- MIAME(
  name = "ex23",
)

In [38]:
ssgsea = ssgseaParam(eset, gsets)

ssgsea_out = gsva(ssgsea)
ssgsea_out_t = t(exprs(ssgsea_out))



ℹ GSVA version 2.0.7

! 96 genes with constant values throughout the samples

! No annotation metadata available in the input expression data object

! Attempting to directly match identifiers in expression data to gene sets

ℹ Calculating  ssGSEA scores for 2 gene sets

ℹ Calculating ranks

ℹ Calculating rank weights

ℹ Normalizing ssGSEA scores

✔ Calculations finished



In [39]:
mg <- merge(meta3, ssgsea_out_t, by = "row.names")
rownames(mg) <- mg$Row.names
mg$Row.names <- NULL

write.csv(mg, 'data/09_externalds/18k_clean/gsva_160pt_IMMPOP.csv')
